In [ ]:
import mlflow.sklearn
import mlflow.xgboost
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,accuracy_score
import xgboost as xgb
import mlflow


# Load the dataset
# Ensure the CSV file is in the same directory or provide the full path
# Replace 'fraud_data.csv' with the actual path to your CSV file

def data_visualization_analysing(data:pd.DataFrame):
    data.head(5)
    #getting the info for the data set
    data.info()
    #checking for imbalance data
    #analysis of the data
    # Null value or NaN check credit.isna().sum()
    print("Null Check",data.isnull().sum())
    #duplicated 
    print("Duplicated",data.duplicated().value_counts())
        

def check_imbalance_data(data:pd.DataFrame,target_feature:str=None)->bool:
    if target_feature:
        class_count=data[target_feature].value_counts()
        class_count=class_count.apply(lambda x:(x/class_count.sum())*100).reset_index()
        if any(class_count.min())<20:
            print("Data is imbalance\n",class_count)
            return True
        else:
            return False
             


def dependent_target_splitter(data:pd.DataFrame,target_feature:str=None,balance:bool=False):
    # data_visualization_analysing(data)
    y=data[target_feature]
    X=data.drop(axis=1,columns=[target_feature])  
    if check_imbalance_data(data,target_feature) and balance:
        X_resampled, y_resampled=get_balanced_data(X,y)
        return X_resampled, y_resampled
    else:
        return X,y
        


def model_logistic_regression(data:list[pd.DataFrame]):
    model=LogisticRegression(random_state=42,max_iter=1000)
    model.fit(X=data[0],y=data[2])
    return model
   
def model_xgboost(data:list[pd.DataFrame]):
    model=xgb.XGBClassifier(tree_method='hist',enable_categorical=True)
    model.fit(X=data[0],y=data[2])
    graph=xgb.to_graphviz(model,num_trees=1)
    # xgb.plot_tree(model)
    # xgb.plot_importance(model)
    return model


def model_random_forest(data:list[pd.DataFrame]):
    model=RandomForestClassifier(criterion='gini')
    model.fit(X=data[0],y=data[2])
    return model
    
def model_prediction(model,data):
    y_test_predict=model.predict(X=data[1])
    y_train_predict=model.predict(X=data[0])
    test_accuracy=accuracy_score(y_test_predict,data[3])
    train_accuracy=accuracy_score(y_train_predict,data[2])
    report=classification_report(y_true=data[3],y_pred=y_test_predict,output_dict=True)
    print("Test Accuracy:", test_accuracy)
    print("Train Accuracy:", train_accuracy)
    print("Classification Report:\n", report)
    return report


def model_training(data: list[pd.DataFrame], models: list):
    mlflow.autolog()
    for model_name in models:
        with mlflow.start_run(run_name=model_name):
            if model_name=="logistic_regression":
                model= model_logistic_regression(data=data)   
            if model_name=="xgboost":
                model= model_xgboost(data=data)   
            if model_name=="random_forest":
                model= model_random_forest(data=data)    
        report=model_prediction(model,data)
        #  mlflow_ingestion(model_name=model_name,model=model,metrics=report)    
        


def load_fraud_data(file:str="creditcard.csv")->pd.DataFrame:
    """
    Load the fraud data from a CSV file.
    
    Returns:
        pd.DataFrame: DataFrame containing the fraud data.
    """
    try:
        fraud_data = pd.read_csv('creditcard.csv')
        return fraud_data
    except FileNotFoundError:
        print("The file 'fraud_data.csv' was not found. Please check the path.")
        return None

def get_balanced_data(X: pd.DataFrame, y: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Balance the dataset using SMOTE and return the resampled X and y.
    """
    smote = SMOTE(sampling_strategy="auto", random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X, y)
    print(f"After resampling-{y_resampled.value_counts()}")
    return X_resampled, y_resampled


def mlflow_ingestion(**kwargs):
    print("Jai Shri Ram",kwargs)
    model_name=kwargs.get('model_name',None)
    model=kwargs.get('model',None)
    params=kwargs.get('params',None)
    metrics=kwargs.get('metrics',None)
        
    with mlflow.start_run(run_name=model_name):
        if model_name=="xgboost":
            mlflow.xgboost.log_model(xgb_model=model,name=model_name,registered_model_name=model_name,input_example=X_train[0:2])
        else:
            mlflow.sklearn.log_model(sk_model=model,name=model_name,registered_model_name=model_name,input_example=X_train[0:2])
        if params:
            mlflow.log_params(params)
        if metrics:  
            metrics={"accuracy":kwargs['metrics']['accuracy'],"precision_zero":kwargs['metrics']['0']['precision'],"recall_zero":kwargs['metrics']['0']['recall'],"precision_one":kwargs['metrics']['1']['precision'],"recall_one":kwargs['metrics']['1']['precision']}  
            mlflow.log_metrics(metrics=metrics)

def mlfow_experiment(experiment:str="mlfow_exp"):
    mlflow.set_tracking_uri(uri='http://127.0.0.1:5000/')
    mlflow.set_experiment(experiment_name=experiment)
    

if __name__=="__main__":
    mlfow_experiment(experiment="fraud_detection")
    data=load_fraud_data()
    X,y=dependent_target_splitter(data,target_feature="Class",balance=True)
    X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,stratify=y,test_size=0.2)  
    model_list_1=['logistic_regression',"xgboost","random_forest"]
    model_list=["xgboost"]
    model_training(data=[X_train,X_test,y_train,y_test],models=model_list_1) 






: 